# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tabeeljohn2/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
import os, subprocess
import pandas as pd, numpy as np
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("/content/flyrank-ml-internship"):
        subprocess.run(["git","clone","https://github.com/tabeeljohn2/flyrank-ml-internship.git","/content/flyrank-ml-internship"])
    os.chdir("/content/flyrank-ml-internship")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(os.getcwd(), df.shape)

/content/flyrank-ml-internship (30000, 44)


In [15]:
!cat skills/training-honest-models/SKILL.md
!grep -n -i -E "leak|window|90d|30d|trend" skills/flyrank/flyrank-data/SKILL.md

---
name: training-honest-models
description: Trains a first model the honest way — method chosen to fit the question, compared against the baseline on the same split and metric, errors read before scores are believed. Use when moving from a rule baseline to a learned model, or when reviewing a model that reports only a single score.
---

# Training honest models

The model is the easy part. The honesty is the work: same data, same split, same metric as the
baseline — then read the errors before believing the score.

## Choose the method to fit the question

| Question shape | Start with | Because |
|---|---|---|
| yes/no with an observed label | Logistic Regression, then Random Forest | readable → stronger |
| "which first?" ranking | any classifier's probability, evaluated at precision@K | ranking needs scores, not labels |
| grouping items | K-Means (pick k with silhouette), then NAME clusters after inspecting them | unsupervised needs human naming |
| "what drives X?" | simple mode

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Task: rank pages by the chance they decline, so editors refresh the right ones first. This is a ranking/scoring task.
Target: declined = 1 if impressions in the last 30 days fell below 70% of the previous 30 days. It is observed, not defined by a rule.
Methods: Logistic Regression (simple, readable), Random Forest (can catch non-linear patterns). I compare both to my Week-4 rule. A more complex model is only better if it wins on the same test pages.
Metrics: ROC-AUC, and precision@K (of the K pages ranked highest, how many really declined), compared with the base rate.

In [16]:
d = df[df["impressions_prev_30d"] >= 50].copy()
d["ratio"] = d["impressions_last_30d"] / d["impressions_prev_30d"]
d["declined"] = (d["ratio"] < 0.7).astype(int)
print("n =", len(d), "| base rate =", round(d["declined"].mean(), 3))

NUM = ["days_since_last_update","content_age_days","word_count","search_volume","competition","cpc"]
CAT = ["content_type","main_intent"]
# Add "impressions_90d","avg_position","ctr" ONLY if the data guide says they end before the last-30d window.
FEATURES = NUM + CAT

n = 20249 | base rate = 0.521


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Pages from the same client are similar, so I split by client_id (grouped split). No client appears in both train and test. Baseline and models use the same test pages.

In [17]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(d, d["declined"], groups=d["client_id"]))
train, test = d.iloc[tr_idx].copy(), d.iloc[te_idx].copy()
print(len(train), len(test), "| shared clients:", len(set(train.client_id) & set(test.client_id)))

17022 3227 | shared clients: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [18]:
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

K = int(0.10 * len(test))          # top 10% of test pages
base_rate = test["declined"].mean()

def prec_at_k(y, score, k=K):
    top = np.argsort(-np.asarray(score))[:k]
    return np.asarray(y)[top].mean()

# Week-4 baseline, computed on the test pages
b = 0.6*test["days_since_last_update"].rank(pct=True) + 0.4*test["impressions_90d"].rank(pct=True)

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT)])

models = {
 "Logistic Regression": Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
 "Random Forest": Pipeline([("pre", pre), ("m", RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=50, random_state=42, n_jobs=-1))]),
}

rows = [{"model":"Week-4 rule (baseline)",
         "AUC": roc_auc_score(test["declined"], b),
         "precision@K": prec_at_k(test["declined"], b)}]
scores = {}
for name, m in models.items():
    m.fit(train[FEATURES], train["declined"])
    s = m.predict_proba(test[FEATURES])[:,1]
    scores[name] = s
    rows.append({"model":name, "AUC": roc_auc_score(test["declined"], s),
                 "precision@K": prec_at_k(test["declined"], s)})

res = pd.DataFrame(rows)
res["base_rate"] = base_rate
res["lift@K"] = res["precision@K"] / base_rate
print("test n =", len(test), "| K =", K)
res.round(3)

test n = 3227 | K = 322


,model,AUC,precision@K,base_rate,lift@K
0,Week-4 rule (baseline),0.360,0.270,0.457,0.591
1,Logistic Regression,0.585,0.705,0.457,1.542
2,Random Forest,0.556,0.596,0.457,1.305


Same test pages (n = 3,227; 20% of clients held out; 0 clients shared with train), same metrics, test base rate = 0.457. The Week-4 rule scores AUC 0.360, below chance. This is likely partly because impressions_90d overlaps the label window. Logistic Regression reaches AUC 0.585 and precision@322 of 0.705 (lift 1.54). Random Forest reaches 0.556 and 0.596 (lift 1.31). Over 10 repeated client splits, mean AUC was 0.420 for the rule (never above 0.487), 0.583 for Logistic Regression and 0.585 for Random Forest. The two models are tied, and the split-to-split spread is large (Logistic Regression ranged from 0.455 to 0.876). In grouped 5-fold CV on the training set the AUCs were 0.527 ± 0.079 and 0.541 ± 0.075. Directionally both models beat the baseline, but the signal is weak and varies by client. I prefer Logistic Regression because Random Forest's extra complexity buys no gain. The "best model" in Section 4 was picked using test AUC, which is mild peeking, but both models are reported.

In [19]:
from sklearn.model_selection import GroupKFold, cross_val_score
for name, m in models.items():
    cv = cross_val_score(m, train[FEATURES], train["declined"], groups=train["client_id"],
                         cv=GroupKFold(n_splits=5), scoring="roc_auc")
    print(name, "CV AUC:", cv.mean().round(3), "+/-", cv.std().round(3))

Logistic Regression CV AUC: 0.527 +/- 0.079
Random Forest CV AUC: 0.541 +/- 0.075


In [20]:
import sklearn; print("sklearn", sklearn.__version__, "| seed 42")

sklearn 1.6.1 | seed 42


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [21]:
from sklearn.inspection import permutation_importance
best = max(models, key=lambda n: roc_auc_score(test["declined"], scores[n]))
pi = permutation_importance(models[best], test[FEATURES], test["declined"],
                            scoring="roc_auc", n_repeats=5, random_state=42)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False)
print("best model:", best); imp.round(4)

best model: Logistic Regression


,0
content_age_days,0.0989
cpc,0.0006
word_count,0.0001
search_volume,-0.0000
competition,-0.0001
main_intent,-0.0010
days_since_last_update,-0.0033
content_type,-0.0034


In [22]:
test["score"] = scores[best]
top = test.sort_values("score", ascending=False).head(K)
fp = top[top["declined"]==0]
print("false positives in top K:", len(fp), "of", K)
print(top.groupby("declined")[NUM].median().round(1))
print(top.groupby("content_type")["declined"].agg(["size","mean"]).round(2))

false positives in top K: 95 of 322
          days_since_last_update  content_age_days  word_count  search_volume  \
declined                                                                        
0                           20.0             144.0      2600.0            0.0   
1                           20.0             141.0      2827.0            0.0   

          competition  cpc  
declined                    
0                 0.0  0.0  
1                 0.0  0.0  
                    size  mean
content_type                  
comparison article   186  0.66
feedly article        11  0.73
keyword article      125  0.77


In [23]:
cols = ["content_id","score","declined","content_age_days","days_since_last_update","content_type"]
print(fp.sort_values("score", ascending=False).head(3)[cols])

                 content_id     score  declined  content_age_days  \
14430  content_a278bb5d581a  0.650964         0               280   
9441   content_1b8802e11084  0.647926         0               118   
15216  content_16bd4fc24d87  0.644889         0                90   

       days_since_last_update     content_type  
14430                      20  keyword article  
9441                       20  keyword article  
15216                      20  keyword article  


In [24]:
tmp = train.copy()
tmp["age_bucket"] = pd.qcut(tmp["content_age_days"], 5, duplicates="drop")
print(tmp.groupby("age_bucket", observed=True)["declined"].agg(["size","mean"]).round(3))

                 size   mean
age_bucket                  
(89.999, 118.0]  3465  0.603
(118.0, 148.0]   3441  0.656
(148.0, 256.0]   3330  0.593
(256.0, 333.0]   3547  0.523
(333.0, 517.0]   3239  0.278


In [25]:
out = []
for seed in range(10):
    g = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    a, c = next(g.split(d, d["declined"], groups=d["client_id"]))
    tr, te = d.iloc[a], d.iloc[c]
    bb = 0.6*te["days_since_last_update"].rank(pct=True) + 0.4*te["impressions_90d"].rank(pct=True)
    row = {"Week-4 rule": roc_auc_score(te["declined"], bb)}
    for n, m in models.items():
        m.fit(tr[FEATURES], tr["declined"])
        row[n] = roc_auc_score(te["declined"], m.predict_proba(te[FEATURES])[:,1])
    out.append(row)
pd.DataFrame(out).agg(["mean","std","min","max"]).round(3)

,Week-4 rule,Logistic Regression,Random Forest
mean,0.420,0.583,0.585
std,0.048,0.125,0.102
min,0.345,0.455,0.484
max,0.487,0.876,0.799


Permutation importance on the test pages shows content_age_days as the only feature that matters (AUC drop 0.099). The others are near zero, and days_since_last_update is slightly negative (-0.003), so staleness adds nothing here, which agrees with my MIXED verdict for Signal 1. Direction: the observed decline rate falls with age (0.603, 0.656, 0.593, 0.523, then 0.278 for pages aged 333 to 517 days), so newer pages decline more. Because one feature carries the whole model, I treat it with caution. I did not verify why age relates to the label, so this is a directional pattern, not a cause. In the top 322 picks, 95 (29.5%) did not decline. The 3 highest-scored wrong picks were all keyword articles with 20 days since update and content age of 280, 118 and 90 days, scored about 0.65. In the top picks the median days_since_last_update is 20 for both right and wrong picks, so staleness does not separate them. By type, keyword articles were right 77% of the time (n = 125), comparison articles 66% (n = 186), and feedly articles 73% (n = 11, too small to trust). Limits: one 60-day window, no seasonality, no time-based split, a decline threshold (0.7) that I chose, and large variation across client splits. These are observed, directional, decision-support results, not proof of cause.

## Self-check

Before you submit, confirm each line honestly:

- [ ✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅ ] No client names, URLs, or private queries anywhere
- [ ✅ ] My claims use careful words: observed, measured, directional, decision-support
- [ ✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.